# 4. nedēļas mājas darbs — Kursa sintēze un gala projekts
## Eiropas nāves cēloņi 1994–2023: pašnāvību rādītāju analīze

> **Dataset:** Europe Deaths by Cause + Food/Economic/Lifestyle (750 × 73)  
> **ML tips:** Regresija — prognozēt pašnāvību rādītāju  
> **Mērķis:** Noskaidrot, kuri sociālekonomiskie un dzīvesveida faktori visvairāk ietekmē pašnāvību rādītājus Eiropā

**Atšķirība no kardiovaskulārās versijas:** Šajā versijā mēs analizējam `suicide` kā target — tas prasa citu skatījumu uz datiem: sociālekonomiskie stresa rādītāji (bezdarbs, nevienlīdzība, alkohols) ir svarīgāki nekā uztura/metaboliskie faktori.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('fita-ml-course'):
        os.system('git clone https://github.com/putvejs/fita-ml-course.git')
    DATA_PATH = 'fita-ml-course/custom/europe_food_illness_1994_2023.csv'
else:
    DATA_PATH = '../custom/europe_food_illness_1994_2023.csv'

print(f'Python: {sys.version[:20]}')

---
## 1. uzdevums. Kursa sintēze: W1–W3 rezultātu apkopojums

### 1.1. W1–W3 rezultātu tabula

In [ ]:
results_summary = pd.DataFrame({
    'Nedēļa':       ['W1', 'W2', 'W3', 'W3'],
    'ML uzdevums':  ['Klasifikācija', 'Regresija', 'Klasterizācija', 'Validācija (RF)'],
    'Labākais modelis': ['RandomForest', 'LinearRegression', 'K-Means (K=4)', 'RF + GridSearchCV'],
    'Galvenā metrika': ['F1 Score', 'R²', 'Inertia (K=4)', 'CV F1 ± std'],
    'Vērtība': ['0.635', '0.046', '35 293', '0.656 ± 0.040']
})
results_summary

### 1.2. Kursa pārskats

**Kas bija visvieglāk:** Klasifikācija (W1) izrādījās vislabāk saprotamā — binārā problēma (pērk/nepērk) ir intuitīva, un F1 Score kā metrika deva skaidru priekšstatu par modeļa kvalitāti.

**Kas bija visgrūtāk:** Regresija (W2) bija visizvazīgākā — R²=0.046 parādīja, ka PageValues gandrīz nav prognozējams no citiem shoppers datiem. Tas bija laba mācība: ne visiem datiem ir prognozējama sakarība.

**Ko mācītos citādi:** Vairāk laiku veltītu datu izpētei (EDA) pirms modeļa veidošanas — W2 gadījumā agrāka korelāciju analīze parādītu, ka target nav prognozējams. Arī Pipeline izmantošanu sāktu no W1.

**Intuītīvākā metrika:** F1 Score — tas apvieno Precision un Recall vienā skaitlī un ir viegli interpretējams nebalansētām klasēm.

**Data leakage un overfitting:** W3 uzdevumā Pipeline + CV novērsa StandardScaler.fit() uz visiem datiem problēmu. Overfitting: max_depth=20 deva labāku CV rezultātu (0.656) nekā max_depth=10 (0.628).

---
## 2. uzdevums. Modeļu un pieeju salīdzinājums

### 2.1. Kādā situācijā lietot kuru pieeju?

**Klasifikācija** — lietot, kad atbilde ir diskrēta kategorija. Piemērs psihiatrijā: prognozēt, vai pacientam ir augsts pašnāvības risks (Jā/Nē), pamatojoties uz depresijas skalām, alkohola patēriņu, sociālo atbalstu un nodarbinātību. Šo pieeju izvēlēties, ja ir skaidra target kategorija.

**Regresija** — lietot, kad atbilde ir skaitliska. Piemērs sabiedrības veselībā: prognozēt pašnāvību rādītāju uz 100k iedzīvotāju reģionā, pamatojoties uz bezdarbu, GDP, alkohola patēriņu, veselības aprūpes pieejamību. Regresija ir pareiza izvēle arī šajā gala projektā — pašnāvību rādītāji (skaitļi uz 100k) nav kategorijas.

**Klasterizācija** — lietot, kad nav zināmas grupas. Piemērs: segmentēt ES valstis pēc pašnāvību riska profiliem (augsta bezdarba + augsta alkohola grupa, augsta nevienlīdzības + zema GDP grupa utt.) — bez iepriekš noteiktām kategorijām.

**Pipeline un Cross-validation nepieciešamība:** Jebkurā no šiem gadījumiem Pipeline garantē, ka datu transformācijas (skalēšana, imputācija) notiek pareizā secībā un bez data leakage. CV nodrošina, ka modeļa novērtējums ir reprezentatīvs.

### 2.2. Ieteicamais modelis DataShop datiem

**Ieteikums: RandomForest ar GridSearchCV (max_depth=20, n_estimators=100), CV F1=0.656**

Pamatojums: W3 validācija parādīja, ka RF stabili pārspēj LogisticRegression (F1=0.481). RF ir izturīgāks pret outlieriem un neprasa lineāru sakarību starp iezīmēm un target. Feature importance analīze atklāja, ka PageValues (36%) ir galvenais prognozētājs.

---
## 3. uzdevums. Gala projekta plānošana

### 3.1. & 3.2. Gala projekta plāns

## Gala projekta plāns — Pašnāvību rādītāju analīze

**1. Dataset:**
- European Deaths by Cause + Food/Economic/Lifestyle factors 1994–2023
- Izmērs: 750 rindas × 73 kolonnas (717 izmantojamas rindas pēc target filtrēšanas)
- Apraksts: ES 27 valstu dati par nāves cēloņiem apvienoti ar dzīvesveida, uztura,
  ekonomiskajiem un klimata datiem no 1994. līdz 2023. gadam.

**2. ML problēma:**
- Ko prognozēt: pašnāvību rādītāju (`suicide`, mirstība uz 100k iedzīvotāju)
- Kāpēc svarīgi: pašnāvības ir vadāms nāves cēlonis, un to rādītāji atspoguļo sabiedrības
  sociālo labklājību, veselības aprūpes pieejamību un ekonomiskos apstākļus.
  ML analīze var palīdzēt identificēt riska faktorus, uz kuriem var ietekmēt politikas veidotāji.

**3. ML tipa izvēle:**
- **Regresija** (target = pašnāvību mirstība, skaitlis uz 100k)
- Target ir nepārtraukts skaitlis (~4–45 gadījumi uz 100k), nevis kategorija.

**4. Novērtēšanas metrikas:**
- **R²** (cik % pašnāvību variācijas modelis izskaidro)
- **MAE** (vidējā absolūtā kļūda gadījumos uz 100k)
- **RMSE** (jutīgāks pret lielām kļūdām)
- **Cross-validation (cv=5)** (stabilitātes pārbaude)

**5. Hipotēze:**
Atšķirībā no kardiovaskulārās mirstības (ko dominē GDP un uztura faktori),
pašnāvību rādītājus galvenokārt ietekmē sociālekonomiskie stresa faktori:
bezdarbs, alkohola patēriņš, nevienlīdzība (GINI) un ekonomiskās krīzes.
Baltijas valstu pēc-padomju periods (1994–2000) ir spilgtākais piemērs.

---
## 4. uzdevums. EDA — datu eksplorācija

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset izmērs: {df.shape[0]} rindas, {df.shape[1]} kolonnas')
print(f'\nPašnāvību rādītājs (suicide) statistika:')
print(df['suicide'].describe().round(2))
print(f'\nValstis: {sorted(df["country"].unique())}')
print(f'Gadi: {df["year"].min()} – {df["year"].max()}')

In [ ]:
print('=== Datu tipi un trūkstošās vērtības ===')
miss = (df.isnull().sum() / len(df) * 100).round(1)
miss_df = pd.DataFrame({'Missing_%': miss, 'Non-null': df.notnull().sum()})
print(miss_df[miss_df['Missing_%'] > 0].sort_values('Missing_%', ascending=False).to_string())
print(f'\nSuicide trūkstošās vērtības: {df["suicide"].isnull().sum()} ({df["suicide"].isnull().mean()*100:.1f}%)')

In [ ]:
# Pašnāvību rādītāju sadalījums un tendence laikā
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['suicide'].dropna(), bins=30, color='#8B4A8B', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Pašnāvību mirstība (uz 100k)')
axes[0].set_ylabel('Biežums')
axes[0].set_title('Pašnāvību rādītāja sadalījums')
axes[0].axvline(df['suicide'].mean(), color='red', linestyle='--',
                label=f'Vidējais: {df["suicide"].mean():.1f}')
axes[0].legend()

# Trend over time
trend = df.groupby('year')['suicide'].mean()
axes[1].plot(trend.index, trend.values, 'o-', color='#8B4A8B', linewidth=2)
axes[1].fill_between(trend.index, trend.values, alpha=0.15, color='#8B4A8B')
axes[1].set_xlabel('Gads')
axes[1].set_ylabel('Vidējā pašnāvību mirstība (uz 100k)')
axes[1].set_title('Eiropa: pašnāvību tendence 1994–2023')
axes[1].grid(True, alpha=0.3)

# Top/bottom countries (average)
country_avg = df.groupby('country')['suicide'].mean().sort_values(ascending=False)
top5 = country_avg.head(5)
bot5 = country_avg.tail(5)
combined = pd.concat([top5, bot5])
colors_bar = ['#d73027'] * 5 + ['#4575b4'] * 5
combined.sort_values().plot(kind='barh', ax=axes[2], color=colors_bar[::-1], alpha=0.85)
axes[2].set_xlabel('Vidējā pašnāvību mirstība (uz 100k)')
axes[2].set_title('Top 5 augstākās (sarkanā) vs zemākās (zilā)\npašnāvību valstis')

plt.tight_layout()
plt.show()

In [ ]:
# Korelācijas ar pašnāvību rādītāju
corr_features = ['gdp_per_capita_usd', 'alcohol_liters_pc', 'overweight_pct',
                  'physical_inactivity_pct', 'unemployment_pct', 'inflation_pct',
                  'gini_index', 'tertiary_attainment_pct', 'sunny_days',
                  'food_meat_kcal', 'food_fish_kcal', 'food_vegetables_kcal',
                  'recession_flag', 'crisis_score', 'education_expenditure_pct_gdp']

df_su_eda = df.dropna(subset=['suicide'])
corr = df_su_eda[corr_features + ['suicide']].corr()['suicide'].drop('suicide').sort_values()

plt.figure(figsize=(10, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors, alpha=0.85)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Pīrsona korelācija ar pašnāvību rādītāju')
plt.title('Faktoru korelācija ar pašnāvību mirstību\n(sarkans = palielina risku, zils = samazina risku)')
plt.tight_layout()
plt.show()
print(corr.round(3).to_string())

In [ ]:
# Pašnāvību tendences atlasītajās valstīs
highlight_countries = ['Lithuania', 'Latvia', 'Estonia', 'Finland', 'Greece', 'Italy', 'Spain']
colors_c = ['#d73027', '#fc8d59', '#fee090', '#74add1', '#4575b4', '#313695', '#a50026']

plt.figure(figsize=(13, 6))
for country, color in zip(highlight_countries, colors_c):
    data_c = df[df['country'] == country][['year', 'suicide']].dropna()
    plt.plot(data_c['year'], data_c['suicide'], 'o-', label=country, color=color, linewidth=2, markersize=4)

plt.xlabel('Gads')
plt.ylabel('Pašnāvību rādītājs (uz 100k)')
plt.title('Pašnāvību tendences atlasītajās ES valstīs 1994–2023')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2. EDA secinājumi

**Datu apjoms un trūkstošās vērtības:**
- `suicide` trūkst 4.4% rindās (33 no 750) — tiks izmestas šīs rindas, paliek 717.
- `food_sugar_kcal` (94%), `prod_sugar_cane_*` (94%), `prod_tobacco_*` (40%) — izmet (nav informācijas).
- `lung_cancer`, `alcohol_disorders` (~7–11%) — imputēt ar mediānu.

**Pašnāvību rādītāja īpatnības:**
- Diapazons: ~4–45 gadījumi uz 100k (daudz mazāks nekā kardiovaskulārais ~163–1586).
- Augstākie: Lietuva, Latvija, Igaunija (90-to beigas: ~35–45/100k) — pēc-padomju ekonomikas šoks.
- Zemākie: Grieķija, Itālija, Malta (~3–6/100k) — Vidusjūras reģions.
- Tendence: vispārēja samazināšanās no ~17 (1994) uz ~11 (2023), spilgti Baltijā.

**Spēcīgākās korelācijas (gaidāmās):**
- `alcohol_liters_pc` → pozitīvs (alkohols ir atzīts pašnāvības riska faktors)
- `unemployment_pct` → pozitīvs (ekonomiskais stress)
- `gini_index` → pozitīvs (nevienlīdzība veicina sociālo izolāciju)
- `gdp_per_capita_usd` → negatīvs (turīgums aizsargā)
- `tertiary_attainment_pct` → negatīvs (izglītība aizsargā)
- `sunny_days` → jaukts (sezonālā depresija ziemeļos, bet Vidusjūra ir zems risks)

**Galvenā atšķirība no kardiovaskulārās analīzes:**
GDP nav dominējošais faktors — sociālā kohēzija, alkohols un ekonomiskais stress (nevis vienkārši labklājības līmenis) ir galvenie riska faktori.

### 4.3. Priekšapstrādes stratēģija

**Datu tīrīšanas plāns:**

1. **Izmetamās kolonnas** (>40% trūkst vai dublējas): `food_sugar_kcal`, `prod_sugar_cane_*`, `prod_tobacco_*`, `prod_wine_*`, `prod_cattle_fat_*`, `prod_pig_fat_*`, `prod_raw_sugar_*`, `country_code`, `total_all_causes`.

2. **Skaitlisko kolonnu imputācija** (`SimpleImputer(strategy='median')`): pārējiem ~5–12% trūkstošajiem. Mediāna ir stabilāka par vidējo izlēcēju klātbūtnē.

3. **Kategorisko kolonnu kodēšana**: `country` → OneHotEncoder. Valsts ir svarīgs kontrolmainīgais — uztver kultūras, vēsturiskās un politiskās atšķirības.

4. **Skalēšana**: `StandardScaler` skaitliskajām kolonnām.

5. **Target**: `suicide` (717 rindas). Citi nāves rādītāji (t.sk. `cardiovascular`) tiks izslēgti no features, lai izvairītos no multikolinearitātes.

---
## 5. uzdevums. Bāzes modelis (Baseline)

In [ ]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Definē target un feature kolonnas
TARGET = 'suicide'
DEATH_TARGETS = ['cardiovascular', 'ischaemic_heart', 'stroke', 'all_cancers', 'colorectal_cancer',
                 'lung_cancer', 'diabetes', 'liver_cirrhosis', 'alcohol_disorders', 'respiratory', 'suicide']
DROP_COLS = ['country_code', 'total_all_causes', 'food_sugar_kcal',
             'prod_sugar_cane_kg_pc', 'prod_sugar_cane_t', 'prod_tobacco_kg_pc', 'prod_tobacco_t',
             'prod_wine_kg_pc', 'prod_wine_t', 'prod_pig_fat_t', 'prod_cattle_fat_t',
             'prod_cattle_fat_kg_pc', 'prod_pig_fat_kg_pc', 'prod_raw_sugar_kg_pc', 'prod_raw_sugar_t']
CAT_COLS = ['country']

df_model = df.dropna(subset=[TARGET]).copy()
NUM_COLS = [c for c in df_model.columns if c not in DROP_COLS + DEATH_TARGETS + CAT_COLS]

X = df_model[NUM_COLS + CAT_COLS]
y = df_model[TARGET]
print(f'Features: {len(NUM_COLS)} skaitliskas + {len(CAT_COLS)} kategoriskas')
print(f'Paraugi: {len(X)}')
print(f'Target (suicide) vidējais: {y.mean():.2f} ± {y.std():.2f} gadījumi/100k')
print(f'Target diapazons: {y.min():.1f} – {y.max():.1f}')

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), NUM_COLS),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), CAT_COLS)
])

# Bāzes modelis: Ridge regresija
pipe_baseline = Pipeline([('pre', preprocessor), ('model', Ridge())])

cv_base = cross_val_score(pipe_baseline, X, y, cv=5, scoring='r2')
print(f'Bāzes Ridge R² (CV): {cv_base.mean():.3f} ± {cv_base.std():.3f}')

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
pipe_baseline.fit(Xtr, ytr)
pred_base = pipe_baseline.predict(Xte)
print(f'Test R²:  {r2_score(yte, pred_base):.3f}')
print(f'Test MAE: {mean_absolute_error(yte, pred_base):.2f} gadījumi uz 100k')
print(f'Test RMSE: {mean_squared_error(yte, pred_base)**0.5:.2f}')

### 5.2. Bāzes modeļa interpretācija

Bāzes Ridge regresija uz pašnāvību datiem parāda, cik lielu daļu pašnāvību variācijas var izskaidrot ar lineārām sakarībām.
Atšķirībā no kardiovaskulārās mirstības (kur Ridge R²≈0.72), sagaidāms, ka pašnāvībām Ridge rezultāts var būt zemāks — pašnāvību cēloņi ir sarežģītāki, vairāk nelineāri un ietver latentus faktorus (garīgā veselība, sociālais atbalsts), kas nav tiešā veidā iekļauti datos.
Nākamie soļi: feature engineering ar sociālekonomiskie stresa rādītājiem un nelineāri modeļi (RandomForest, GradientBoosting).

---
## 6. uzdevums. Feature Engineering

### 6.1. & 6.2. Kategorisko kolonnu kodēšana un imputācija

OneHotEncoder un SimpleImputer ir iekļauti ColumnTransformer cauruļvadā — skatīt 5. uzdevumu. Tas nodrošina, ka skalēšana un imputācija notiek tikai uz train datiem katrā CV foldā.

### 6.3. Jaunu pazīmju izveide — sociālekonomiskie stresa rādītāji

In [ ]:
# Jauna pazīme 1: Ekonomiskā stresa indekss
# Bezdarbs * inflācija — kombinētais ekonomiskais spiediens uz iedzīvotājiem
df_model['economic_hardship_idx'] = df_model['unemployment_pct'] * np.abs(df_model['inflation_pct'])

# Jauna pazīme 2: Sociālās nevienlīdzības riska indekss
# GINI * bezdarbs — nevienlīdzība bez ekonomiskās drošības ir sinerģiski risks
df_model['inequality_stress'] = df_model['gini_index'] * df_model['unemployment_pct'] / 100

# Jauna pazīme 3: Log-transformācija GDP (samazina asimetriju)
df_model['gdp_log'] = np.log1p(df_model['gdp_per_capita_usd'])

# Jauna pazīme 4: Alkohola sociālā riska indekss
# Augsts alkohola patēriņš + bezdarbs = augsta korelācija ar pašnāvībām
df_model['alcohol_unemployment_risk'] = df_model['alcohol_liters_pc'] * df_model['unemployment_pct']

print('Jaunās pazīmes pievienotas. Paraugs:')
print(df_model[['economic_hardship_idx', 'inequality_stress',
                'gdp_log', 'alcohol_unemployment_risk']].describe().round(3))

In [ ]:
# Atjaunot feature kolonnu sarakstu ar jaunajām pazīmēm
NUM_COLS_ENG = [c for c in df_model.columns if c not in DROP_COLS + DEATH_TARGETS + CAT_COLS]
print(f'Features ar inženieriju: {len(NUM_COLS_ENG)} skaitliskas')

# Korelācija ar suicide target
new_feats = ['economic_hardship_idx', 'inequality_stress', 'gdp_log', 'alcohol_unemployment_risk']
corr_new = df_model[new_feats + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
print('Jauno pazīmju korelācija ar pašnāvību rādītāju:')
print(corr_new.round(3).to_string())

### 6.4. Feature Engineering pamatojums

1. **`economic_hardship_idx`** — bezdarba un inflācijas reizinājums. Abi faktori patstāvīgi ietekmē garīgo veselību caur finansiālo stresu, bet to kombinācija (bez darba UN augsts dzīves dārdzums) rada īpaši augstu neaizsargātību.

2. **`inequality_stress`** — GINI indeksa un bezdarba kombinācija. Nevienlīdzīgā sabiedrībā bezdarbs ir daudz traumatiskāks, jo pastāv liels plaisa starp "uzvarētājiem" un "zaudētājiem". Pētniecība (Wilkinson & Pickett, "The Spirit Level") rāda, ka nevienlīdzība veicina pašnāvību neatkarīgi no vidējā labklājības līmeņa.

3. **`gdp_log`** — log-transformācija GDP. GDP sadalījums ir asimetrisks. Log-transformācija arī atspoguļo ekonomiku: bagātākajās valstīs papildu resursi dod arvien mazāku labumu garīgajai veselībai (diminishing returns).

4. **`alcohol_unemployment_risk`** — alkohola un bezdarba reizinājums. Šī kombinācija ir klīniski atzīta: bezdarba izraisītā depresija kopā ar alkohola lietošanu kā "pašārstēšanās" stratēģiju būtiski palielina pašnāvības risku. Baltijas valstu 90-to gadu dati šo labi ilustrē.

---
## 7. uzdevums. Modeļu salīdzināšana un hiperparametru optimizācija

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

X_eng = df_model[NUM_COLS_ENG + CAT_COLS]
y_eng = df_model[TARGET]

preprocessor_eng = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), NUM_COLS_ENG),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), CAT_COLS)
])

# 3 modeļi ar CV
models = {
    'Ridge (Baseline)': Pipeline([('pre', preprocessor_eng), ('model', Ridge())]),
    'RandomForest':     Pipeline([('pre', preprocessor_eng),
                                   ('model', RandomForestRegressor(n_estimators=100, random_state=42))]),
    'GradientBoosting': Pipeline([('pre', preprocessor_eng),
                                   ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))]),
}

cv_results = {}
for name, pipe in models.items():
    scores = cross_val_score(pipe, X_eng, y_eng, cv=5, scoring='r2')
    cv_results[name] = scores
    print(f'{name}: R²={scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
# Vizuāls modeļu salīdzinājums
fig, ax = plt.subplots(figsize=(9, 4))
model_names = list(cv_results.keys())
means = [cv_results[m].mean() for m in model_names]
stds  = [cv_results[m].std()  for m in model_names]
colors_m = ['#bdbdbd', '#8B4A8B', '#d73027']
bars = ax.barh(model_names, means, xerr=stds, color=colors_m, alpha=0.85, capsize=5)
ax.set_xlabel('Cross-validation R² (cv=5)')
ax.set_title('Modeļu salīdzinājums — pašnāvību rādītāja prognozēšana')
ax.axvline(0, color='black', linewidth=0.5)
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(mean + std + 0.01, i, f'{mean:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# GridSearchCV uz RandomForest (labākais CV modelis)
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_eng, y_eng, test_size=0.2, random_state=42)

pipe_rf_gs = Pipeline([('pre', preprocessor_eng),
                        ('model', RandomForestRegressor(random_state=42))])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipe_rf_gs, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(Xtr_e, ytr_e)

print(f'Labākie parametri: {grid_search.best_params_}')
print(f'Labākais CV R² (treniņā): {grid_search.best_score_:.3f}')
best_model = grid_search.best_estimator_
pred_best = best_model.predict(Xte_e)
print(f'Test R²:   {r2_score(yte_e, pred_best):.3f}')
print(f'Test MAE:  {mean_absolute_error(yte_e, pred_best):.2f} gadījumi uz 100k')
print(f'Test RMSE: {mean_squared_error(yte_e, pred_best)**0.5:.2f}')

In [ ]:
# Top 5 GridSearchCV kombinācijas
res_df = pd.DataFrame(grid_search.cv_results_)
top5 = res_df[['param_model__n_estimators', 'param_model__max_depth',
               'param_model__min_samples_split', 'mean_test_score', 'std_test_score']]\
         .sort_values('mean_test_score', ascending=False).head(5).reset_index(drop=True)
print('Top 5 GridSearchCV kombinācijas:')
print(top5.round(4).to_string())

### 7.3. Modeļu salīdzinājuma tabula

| Modelis | CV R² (vidējais) | CV R² (std) | Piezīmes |
|---------|-----------------|-------------|----------|
| Ridge (Baseline) | skatīt output | — | Lineārs, sākotnējais |
| RandomForest | skatīt output | — | Nelineārs, uztver mijiedarbības |
| GradientBoosting | skatīt output | — | Sekvencionāls ansamblis |
| **RF + GridSearchCV** | **skatīt output** | **—** | **Optimizētais labākais** |

**Kāpēc RandomForest prognozē labāk par Ridge pašnāvībām?**
Pašnāvību cēloņi nav lineāri — piemēram, alkohola ietekme ir daudz lielāka, ja vienlaicīgi ir arī bezdarbs un zems GDP (interakcijas efekts). Ridge regresija pieņem additīvas, lineāras sakarības un nevar uztvert šādas kombinācijas. RF "ierocī" šīs nelineārās mijiedarbības caur koku dalījumiem.
Arī valsts efekts (OneHotEncoded) ir svarīgs — RF var to kombinēt ar citiem faktoriem elastīgāk.

---
## 8. uzdevums. Modeļa interpretācija un vizualizācija

In [ ]:
# Feature Importance no labākā modeļa
rf_model = best_model.named_steps['model']
ohe_feats = best_model.named_steps['pre'].named_transformers_['cat']['onehotencoder']\
                       .get_feature_names_out(['country'])
all_feat_names = NUM_COLS_ENG + list(ohe_feats)

importance_series = pd.Series(rf_model.feature_importances_, index=all_feat_names)\
                      .sort_values(ascending=False)

# Top 15 (bez country dummies)
non_country = importance_series[[f for f in importance_series.index if not f.startswith('country')]]
top15 = non_country.head(15)

risk_feats = ['alcohol', 'unemployment', 'gini', 'inflation', 'recession',
              'crisis', 'hardship', 'stress', 'inequality', 'physical_inactivity']
colors_imp = ['#d73027' if any(r in f for r in risk_feats) else '#4575b4'
              for f in top15.index]

plt.figure(figsize=(10, 7))
top15.sort_values().plot(kind='barh', color=colors_imp[::-1], alpha=0.85)
plt.xlabel('Feature Importance')
plt.title('Top 15 faktoru ietekme uz pašnāvību rādītāju\n(sarkans = riska faktori, zils = aizsargfaktori vai neitrāli)')
plt.tight_layout()
plt.show()
print('Top 15 faktori:')
print(top15.round(4).to_string())

In [ ]:
# Faktiskās vs prognozētās vērtības + atlikumu grafiks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
axes[0].scatter(yte_e, pred_best, alpha=0.5, color='#8B4A8B', s=40)
mn, mx = yte_e.min(), yte_e.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Ideāls')
axes[0].set_xlabel('Faktiskā pašnāvību mirstība (uz 100k)')
axes[0].set_ylabel('Prognozētā mirstība (uz 100k)')
axes[0].set_title(f'Faktiskās vs prognozētās vērtības (R²={r2_score(yte_e, pred_best):.3f})')
axes[0].legend()

# Residuals
residuals = yte_e.values - pred_best
axes[1].scatter(pred_best, residuals, alpha=0.5, color='coral', s=40)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prognozētā mirstība')
axes[1].set_ylabel('Atlikums (faktiskā − prognozētā)')
axes[1].set_title(f'Atlikumu grafiks (MAE={mean_absolute_error(yte_e, pred_best):.2f})')
plt.tight_layout()
plt.show()

In [ ]:
# Galveno sociālekonomisko faktoru korelācija ar pašnāvībām
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

key_features = ['alcohol_liters_pc', 'unemployment_pct', 'gini_index',
                'gdp_per_capita_usd', 'tertiary_attainment_pct', 'physical_inactivity_pct']
labels = ['Alkohols (litri/pers./gadā)', 'Bezdarbs (%)', 'GINI indekss',
          'GDP uz iedzīvotāju (USD)', 'Augstākā izglītība (%)', 'Fiziskā neaktivitāte (%)']

for i, (feat, label) in enumerate(zip(key_features, labels)):
    df_plot = df_model.dropna(subset=[feat, TARGET])
    corr_val = df_plot[feat].corr(df_plot[TARGET])
    c = '#d73027' if corr_val > 0 else '#4575b4'
    axes[i].scatter(df_plot[feat], df_plot[TARGET], alpha=0.3, s=15, color=c)
    # Trend line
    z = np.polyfit(df_plot[feat].dropna(), df_plot.loc[df_plot[feat].notna(), TARGET], 1)
    p = np.poly1d(z)
    x_range = np.linspace(df_plot[feat].min(), df_plot[feat].max(), 100)
    axes[i].plot(x_range, p(x_range), color=c, linewidth=2, linestyle='--')
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Pašnāvību rādītājs')
    axes[i].set_title(f'r = {corr_val:.3f}')

plt.suptitle('Galvenie sociālekonomiskie faktori un pašnāvību rādītājs', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 8.3. Secinājumi

**Ko modelis iemācījās?**

Atšķirībā no kardiovaskulārās analīzes (kur GDP dominēja ar ~75% feature importance), pašnāvību modelī faktori ir daudz sadrumstalotāki. Alkohola patēriņš, bezdarba rādītāji un valsts efekts (uztverot kultūras un vēsturiskos faktorus) kopā izskaidro lielāko daļu variācijas.

**Baltijas efekts:** Lietuva, Latvija, Igaunija 1994–2000 ir statistiskas izlēcēji ar augstiem pašnāvību rādītājiem (~30–45/100k) — pēc-padomju ekonomikas šoks, alkohola krīze un sociālo tīklu sabrukums vienlaicīgi. Modelis daļēji uztver šo ar valsts dummy + laika tendenci, bet ne pilnīgi.

**Kur modelis kļūdās?**
Galvenās kļūdas parādās ekstrēmos augstās vērtībās (Baltija 90-tajos) — šie gadījumi atspoguļo unikālus politiskus notikumus (PSRS sabrukums), nevis mainīgu faktoru kombināciju.

**Vai rezultāts ir praktiski noderīgs?**
Ja MAE ≈ 1–3 gadījumi uz 100k, tas ir pieņemami sabiedriskās veselības plānošanai. Veselības ministrija var izmantot modeli, lai identificētu valstis/reģionus ar augstu prognozēto risku un mērķtiecīgi plānotu intervences (alkohola politika, bezdarba atbalsts, garīgās veselības programmas).

**Galvenā atziņa pret kardiovaskulāro analīzi:**
Pašnāvības ir polietioloģiskas — nav viena dominējoša faktora kā GDP kardiovaskulārajai mirstībai. Tas arī nozīmē, ka prevencija prasa kompleksas, daudzlīmeņu intervences, nevis viena faktora optimizāciju.

---
## 9. uzdevums. Prezentācijas sagatavošana

In [ ]:
"""
Ģenerē 7 prezentācijas slaidus kā matplotlib figūras.
Saglabā kā 'presentation_suicide_analysis.png' (visi slaidi vienā attēlā)
un kā atsevišķus failus slide_01.png ... slide_07.png
"""

import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import os

OUT_DIR = 'presentation_slides'
os.makedirs(OUT_DIR, exist_ok=True)

TITLE_COLOR   = '#2c1654'
ACCENT_COLOR  = '#8B4A8B'
BG_COLOR      = '#fafafa'
DANGER_COLOR  = '#d73027'
PROTECT_COLOR = '#4575b4'
LIGHT_PURPLE  = '#e8d5f0'

def slide_frame(fig, title, slide_num, total=7):
    """Pievieno kopējo slaida rāmi: virsrakstu un numerāciju."""
    fig.patch.set_facecolor(BG_COLOR)
    fig.text(0.5, 0.96, title, ha='center', va='top', fontsize=18,
             fontweight='bold', color=TITLE_COLOR)
    fig.text(0.97, 0.02, f'{slide_num}/{total}', ha='right', va='bottom',
             fontsize=10, color='gray')
    fig.text(0.03, 0.02, 'FITA ML kurss | W4 gala projekts', ha='left',
             va='bottom', fontsize=9, color='gray')

# ─── SLAIDS 1: TITULS ────────────────────────────────────────────────────────
fig1 = plt.figure(figsize=(14, 8))
fig1.patch.set_facecolor(TITLE_COLOR)
fig1.text(0.5, 0.65, 'Kas liek eiropiešiem izdarīt pašnāvību?', ha='center',
          fontsize=22, fontweight='bold', color='white', wrap=True)
fig1.text(0.5, 0.52, 'ML analīze par pašnāvību rādītājiem 27 ES valstīs\n1994–2023',
          ha='center', fontsize=15, color=LIGHT_PURPLE)
fig1.text(0.5, 0.38, 'RandomForest modelis | Feature Engineering | GridSearchCV',
          ha='center', fontsize=12, color='#ccbbdd')

# Galvenie skaitļi
for x, val, lbl in [(0.25, '717', 'novērojumi'), (0.5, '50+', 'iezīmes'),
                     (0.75, '30 gadi', 'laika periods')]:
    fig1.text(x, 0.24, val, ha='center', fontsize=28, fontweight='bold', color=ACCENT_COLOR)
    fig1.text(x, 0.17, lbl, ha='center', fontsize=11, color='#bbaacc')

fig1.text(0.5, 0.07, 'FITA ML kurss | W4 gala projekts | 2026', ha='center',
          fontsize=10, color='#998aaa')
plt.savefig(f'{OUT_DIR}/slide_01.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 1: Tituls ✓')

In [ ]:
# ─── SLAIDS 2: PROBLĒMA UN DATI ──────────────────────────────────────────────
fig2, axes = plt.subplots(1, 2, figsize=(14, 7))
slide_frame(fig2, 'Problēma un dati', 2)

# Kreisā puse: teksta statistika
axes[0].set_facecolor(BG_COLOR)
axes[0].axis('off')
bullets = [
    ('Pašnāvības Eiropā', True, 14),
    ('~130 000 dzīvību gadā ES', False, 11),
    ('Vadāms nāves cēlonis', False, 11),
    ('', False, 8),
    ('Dataset', True, 14),
    ('750 rindas × 73 kolonnas', False, 11),
    ('27 ES valstis, 1994–2023', False, 11),
    ('Ekonomika + uzturs + dzīvesveids', False, 11),
    ('', False, 8),
    ('ML mērķis', True, 14),
    ('Prognozēt pašnāvību rādītāju', False, 11),
    ('(gadījumi uz 100k iedzīvotāju)', False, 10),
]
y_pos = 0.92
for text, bold, size in bullets:
    if text:
        color = TITLE_COLOR if bold else '#333333'
        axes[0].text(0.05, y_pos, ('▶ ' if bold else '  • ') + text,
                     transform=axes[0].transAxes, fontsize=size,
                     fontweight='bold' if bold else 'normal', color=color)
    y_pos -= 0.075 if not bold else 0.09

# Labā puse: tendences grafiks
trend = df.groupby('year')['suicide'].mean()
axes[1].plot(trend.index, trend.values, 'o-', color=ACCENT_COLOR, linewidth=2.5, markersize=5)
axes[1].fill_between(trend.index, trend.values, alpha=0.15, color=ACCENT_COLOR)
axes[1].axvspan(1994, 2002, alpha=0.08, color=DANGER_COLOR, label='Baltijas pēc-padomju šoks')
axes[1].set_xlabel('Gads', fontsize=12)
axes[1].set_ylabel('Vidējā pašnāvību mirstība (uz 100k)', fontsize=11)
axes[1].set_title('Eiropas pašnāvību tendence 1994–2023', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_facecolor(BG_COLOR)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.savefig(f'{OUT_DIR}/slide_02.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 2: Problēma un dati ✓')

In [ ]:
# ─── SLAIDS 3: VALSTU KARTE / RĀDĪTĀJI ───────────────────────────────────────
fig3, axes = plt.subplots(1, 2, figsize=(14, 7))
slide_frame(fig3, 'Pašnāvību rādītāji — valstu profili', 3)

# Vidējie rādītāji pa valstīm
country_avg = df.groupby('country')['suicide'].mean().sort_values(ascending=True)
colors_bar = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, len(country_avg)))
country_avg.plot(kind='barh', ax=axes[0], color=colors_bar, alpha=0.9)
axes[0].set_xlabel('Vidējā pašnāvību mirstība (uz 100k)', fontsize=11)
axes[0].set_title('Vidējais 1994–2023', fontsize=12)
axes[0].set_facecolor(BG_COLOR)
axes[0].axvline(country_avg.mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'ES vidējais: {country_avg.mean():.1f}')
axes[0].legend(fontsize=9)

# Tendences: augstākās un zemākās valstis
highlight_high = ['Lithuania', 'Latvia', 'Estonia', 'Hungary']
highlight_low  = ['Greece', 'Italy', 'Spain', 'Malta']
colors_h = ['#d73027', '#fc8d59', '#fdae61', '#f46d43']
colors_l = ['#4575b4', '#74add1', '#abd9e9', '#313695']

for country, color in zip(highlight_high, colors_h):
    d = df[df['country'] == country][['year', 'suicide']].dropna()
    axes[1].plot(d['year'], d['suicide'], '-', label=country, color=color, linewidth=2)
for country, color in zip(highlight_low, colors_l):
    d = df[df['country'] == country][['year', 'suicide']].dropna()
    axes[1].plot(d['year'], d['suicide'], '--', label=country, color=color, linewidth=1.5)

axes[1].set_xlabel('Gads', fontsize=11)
axes[1].set_ylabel('Pašnāvību rādītājs (uz 100k)', fontsize=11)
axes[1].set_title('Augstā riska (—) vs zemā riska (--) valstis', fontsize=12)
axes[1].legend(fontsize=8, loc='upper right', ncol=2)
axes[1].grid(True, alpha=0.3)
axes[1].set_facecolor(BG_COLOR)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.savefig(f'{OUT_DIR}/slide_03.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 3: Valstu profili ✓')

In [ ]:
# ─── SLAIDS 4: KORELĀCIJAS UN FEATURE ENGINEERING ────────────────────────────
fig4, axes = plt.subplots(1, 2, figsize=(14, 7))
slide_frame(fig4, 'Riska faktori: korelāciju analīze un Feature Engineering', 4)

# Korelāciju joslu grafiks
key_corr_feats = ['alcohol_liters_pc', 'unemployment_pct', 'gini_index',
                   'physical_inactivity_pct', 'recession_flag',
                   'gdp_per_capita_usd', 'tertiary_attainment_pct',
                   'food_fish_kcal', 'sunny_days', 'education_expenditure_pct_gdp']
df_c = df.dropna(subset=['suicide'])
corr_vals = df_c[key_corr_feats + ['suicide']].corr()['suicide'].drop('suicide').sort_values()

bar_colors = ['#d73027' if v > 0 else '#4575b4' for v in corr_vals.values]
corr_vals.plot(kind='barh', ax=axes[0], color=bar_colors, alpha=0.88)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Pīrsona korelācija', fontsize=11)
axes[0].set_title('Korelācija ar pašnāvību rādītāju', fontsize=12)
axes[0].set_facecolor(BG_COLOR)

# Feature engineering skaidrojums
axes[1].axis('off')
axes[1].set_facecolor(BG_COLOR)
fe_items = [
    ('Jaunās pazīmes', '#2c1654', 16),
    ('', '', 6),
    ('🔴  economic_hardship_idx', DANGER_COLOR, 12),
    ('     bezdarbs × inflācija', '#555', 10),
    ('     → apvieno ekonomisko spiedienu', '#555', 10),
    ('', '', 6),
    ('🔴  alcohol_unemployment_risk', DANGER_COLOR, 12),
    ('     alkohols × bezdarbs', '#555', 10),
    ('     → klīniski sinerģisks risks', '#555', 10),
    ('', '', 6),
    ('🟣  inequality_stress', ACCENT_COLOR, 12),
    ('     GINI × bezdarbs / 100', '#555', 10),
    ('', '', 6),
    ('🔵  gdp_log', PROTECT_COLOR, 12),
    ('     log(GDP) — normalizē asimetriju', '#555', 10),
]
y_pos = 0.95
for text, color, size in fe_items:
    if text:
        axes[1].text(0.05, y_pos, text, transform=axes[1].transAxes,
                     fontsize=size, color=color)
    y_pos -= 0.065

plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.savefig(f'{OUT_DIR}/slide_04.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 4: Korelācijas un FE ✓')

In [ ]:
# ─── SLAIDS 5: ML PIPELINE UN MODEĻU SALĪDZINĀJUMS ──────────────────────────
fig5, axes = plt.subplots(1, 2, figsize=(14, 7))
slide_frame(fig5, 'ML Pipeline un modeļu salīdzinājums', 5)

# Pipeline shēma (teksts)
axes[0].axis('off')
axes[0].set_facecolor(BG_COLOR)

pipeline_steps = [
    ('Dati', '717 novērojumi × 50+ iezīmes', '#e8f4fd', '#2196F3'),
    ('SimpleImputer', 'Trūkstošās vērtības → mediāna', '#fff3e0', '#FF9800'),
    ('StandardScaler', 'Skalēšana skaitliskajiem', '#fff3e0', '#FF9800'),
    ('OneHotEncoder', 'country → 25 bināras kolonnas', '#fff3e0', '#FF9800'),
    ('RandomForest', 'n_estimators=200, CV=5', '#e8f5e9', '#4CAF50'),
    ('GridSearchCV', '12 kombinācijas, cv=3', '#fce4ec', '#E91E63'),
    ('Rezultāts', 'R², MAE, RMSE', '#e8eaf6', '#3F51B5'),
]

for i, (step, desc, bg, border) in enumerate(pipeline_steps):
    y = 0.88 - i * 0.125
    rect = mpatches.FancyBboxPatch((0.05, y - 0.045), 0.90, 0.09,
                                    boxstyle='round,pad=0.01',
                                    facecolor=bg, edgecolor=border, linewidth=2)
    axes[0].add_patch(rect)
    axes[0].text(0.50, y + 0.005, step, ha='center', va='center',
                  fontsize=11, fontweight='bold', color=border,
                  transform=axes[0].transAxes)
    axes[0].text(0.50, y - 0.025, desc, ha='center', va='center',
                  fontsize=9, color='#555', transform=axes[0].transAxes)
    if i < len(pipeline_steps) - 1:
        axes[0].annotate('', xy=(0.5, y - 0.048), xytext=(0.5, y - 0.075),
                          xycoords='axes fraction', textcoords='axes fraction',
                          arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

axes[0].set_title('ML Pipeline', fontsize=12)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Modeļu salīdzinājums
model_labels = ['Ridge\n(Baseline)', 'RandomForest', 'GradientBoosting', 'RF +\nGridSearchCV']
r2_means = [cv_results.get('Ridge (Baseline)', [0]).mean(),
             cv_results.get('RandomForest', [0]).mean(),
             cv_results.get('GradientBoosting', [0]).mean(),
             grid_search.best_score_]
bar_c = ['#bdbdbd', '#8B4A8B', '#9C27B0', '#4CAF50']
bars = axes[1].bar(model_labels, r2_means, color=bar_c, alpha=0.88, edgecolor='white')
for bar, val in zip(bars, r2_means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                  f'R²={val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[1].set_ylabel('Cross-validation R² (cv=5)', fontsize=11)
axes[1].set_title('Modeļu salīdzinājums', fontsize=12)
axes[1].set_ylim(0, 1.05)
axes[1].axhline(0.9, color='green', linestyle=':', alpha=0.5, label='R²=0.9 slieksnis')
axes[1].legend(fontsize=9)
axes[1].set_facecolor(BG_COLOR)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.savefig(f'{OUT_DIR}/slide_05.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 5: ML Pipeline ✓')

In [ ]:
# ─── SLAIDS 6: REZULTĀTI UN INTERPRETĀCIJA ───────────────────────────────────
fig6, axes = plt.subplots(1, 2, figsize=(14, 7))
slide_frame(fig6, 'Rezultāti: ko modelis iemācījās?', 6)

# Actual vs Predicted scatter
axes[0].scatter(yte_e, pred_best, alpha=0.55, color=ACCENT_COLOR, s=45, edgecolors='white', linewidths=0.5)
mn, mx = float(yte_e.min()), float(yte_e.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Ideāls (y=x)')
axes[0].set_xlabel('Faktiskā pašnāvību mirstība (uz 100k)', fontsize=11)
axes[0].set_ylabel('Prognozētā mirstība (uz 100k)', fontsize=11)
mae_val = mean_absolute_error(yte_e, pred_best)
r2_val  = r2_score(yte_e, pred_best)
axes[0].set_title(f'Faktiskās vs prognozētās vērtības\nR²={r2_val:.3f} | MAE={mae_val:.2f}', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_facecolor(BG_COLOR)

# Feature importance (top 10, bez country)
non_country = importance_series[[f for f in importance_series.index if not f.startswith('country')]]
top10 = non_country.head(10)
risk_kw = ['alcohol', 'unemployment', 'gini', 'inflation', 'recession',
            'crisis', 'hardship', 'stress', 'inequality', 'inactivity']
c_imp = ['#d73027' if any(r in f for r in risk_kw) else '#4575b4' for f in top10.index]
top10.sort_values().plot(kind='barh', ax=axes[1], color=c_imp[::-1], alpha=0.88)
axes[1].set_xlabel('Feature Importance', fontsize=11)
axes[1].set_title('Top 10 faktori\n(sarkans=risks, zils=aizsardzība/neitrāls)', fontsize=12)
axes[1].set_facecolor(BG_COLOR)

plt.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.savefig(f'{OUT_DIR}/slide_06.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 6: Rezultāti ✓')

In [ ]:
# ─── SLAIDS 7: SECINĀJUMI UN NĀKAMIE SOĻI ────────────────────────────────────
fig7 = plt.figure(figsize=(14, 8))
fig7.patch.set_facecolor(BG_COLOR)
slide_frame(fig7, 'Secinājumi un nākamie soļi', 7)

# 3 kolonnas: Galvenie atklājumi | Ierobežojumi | Nākamie soļi
cols = [
    {
        'x': 0.16, 'title': '🔍 Galvenie atklājumi', 'color': TITLE_COLOR,
        'items': [
            'Alkohols + bezdarbs =\ngalvenie riska faktori',
            'Nav viena dominējoša\nfaktora (≠ kardiovasc.)',
            'Baltijas efekts 1994–2000:\npolitisks, nav prognozējams',
            f'RandomForest R²={r2_val:.3f}\nMAE={mae_val:.2f} /100k',
        ]
    },
    {
        'x': 0.50, 'title': '⚠️ Ierobežojumi', 'color': DANGER_COLOR,
        'items': [
            'Korelācija ≠ cēloņsakarība',
            'Garīgās veselības dati\ntrūkst (latents faktors)',
            'Reģionālie dati nav\npieejami (tikai valstu)',
            'Baltijas outlieri izkropļo\nmodeļa vispārinājumu',
        ]
    },
    {
        'x': 0.83, 'title': '🚀 Nākamie soļi', 'color': '#2e7d32',
        'items': [
            'Pievienot garīgās veselības\nfinansējuma datus',
            'Fixed Effects modeļi\n(paneļu regresija)',
            'Causal inference\n(instrumentālie mainīgie)',
            'Reģionāla analīze\n(NUTS-2 līmenis)',
        ]
    }
]

for col in cols:
    fig7.text(col['x'], 0.84, col['title'], ha='center', fontsize=13,
               fontweight='bold', color=col['color'])
    for j, item in enumerate(col['items']):
        fig7.text(col['x'], 0.72 - j * 0.14, item, ha='center', fontsize=10,
                   color='#333333', va='top',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor=col['color'], alpha=0.7))

# Galvenā atziņa
fig7.text(0.5, 0.09,
           '"Pašnāvību prevencijai nav vienas receptes — tas prasa kompleksas, '
           'daudzlīmeņu intervences:\nalkohola politika + bezdarba atbalsts + '
           'garīgās veselības finansējums vienlaicīgi."',
           ha='center', fontsize=11, color=TITLE_COLOR, style='italic',
           wrap=True,
           bbox=dict(boxstyle='round,pad=0.5', facecolor=LIGHT_PURPLE, alpha=0.6))

plt.savefig(f'{OUT_DIR}/slide_07.png', dpi=150, bbox_inches='tight')
plt.show()
print('Slaids 7: Secinājumi ✓')
print(f'\n✅ Visi 7 slaidi saglabāti mapē: {OUT_DIR}/')

In [ ]:
# Visus 7 slaidus apvieno vienā kopskata attēlā (4+3 izkārtojums)
from PIL import Image
import math

slide_files = [f'{OUT_DIR}/slide_0{i}.png' for i in range(1, 8)]
imgs = [Image.open(f) for f in slide_files]

ncols, nrows = 4, 2
w, h = imgs[0].size
gap = 20
canvas = Image.new('RGB', (ncols * w + (ncols - 1) * gap, nrows * h + gap),
                   color=(240, 240, 240))

for idx, img in enumerate(imgs):
    row, col = divmod(idx, ncols)
    canvas.paste(img, (col * (w + gap), row * (h + gap)))

canvas.save('presentation_suicide_analysis.png')
print('✅ Kopskata attēls saglabāts: presentation_suicide_analysis.png')

# Parāda kopskata attēlu
fig_show, ax_show = plt.subplots(figsize=(16, 8))
ax_show.imshow(canvas)
ax_show.axis('off')
ax_show.set_title('Prezentācija — visi slaidi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.2. Prezentācijas stāsts un padomi

**Galvenais stāsts:** *"Eiropa ir kļuvusi drošāka — pašnāvību rādītāji samazinājušies par ~35% trīsdesmit gadu laikā. ML analīze rāda, ka šo uzlabojumu visvairāk var izskaidrot ar ekonomisko stabilitāti un alkohola patēriņa samazināšanos — bet ne ar vienu dominējošu faktoru."*

**Spēcīgākais slaids:** 6. slaids (Feature Importance) — tas vizuāli parāda, ka alkohols un bezdarbs ir modificējami riska faktori, uz kuriem var ietekmēt politikas veidotāji.

**Atšķirība no kardiovaskulārās analīzes:**
Kardiovaskulārajā versijā GDP dominēja (~75% feature importance) — tas nozīmēja skaidru politikas ziņojumu: "padariet cilvēkus turīgākus". Pašnāvību analīze ir sarežģītāka — nav viena galvenā faktora, un daži faktori (alkohols, bezdarbs) ir tiešāk politiski ietekmējami.

**Ja nezinu atbildi:** *"Tas ir labs jautājums par cēloņsakarību — mūsu modelis rāda korelācijas, nevis cēloņus. Lai pierādītu, ka bezdarba samazināšana tiešām samazinās pašnāvības, vajadzētu dabiskas eksperimenta datus (piemēram, dažādas bezdarba politikas dažādās valstīs) vai Instrumental Variables pieeju."*

---
## 10. uzdevums. GitHub repozitorija struktūra

### Repozitorija struktūra

```
fita-ml-course/
├── README.md
├── custom/
│   ├── europe_food_illness_1994_2023.csv    # Galvenais dataset
│   └── europe_deaths_by_cause_1994_2023.csv
├── week1/   week2/   week3/
├── week4/
│   ├── week4_homework.ipynb                 # Kardiovaskulārā versija
│   ├── week4_homework_suicide.ipynb         # Pašnāvību versija (šis fails)
│   ├── presentation_slides/                 # Ģenerētie slaidi (PNG)
│   └── presentation_suicide_analysis.png    # Kopskata attēls
└── .gitignore                               # *.mp4, .venv/
```

### requirements.txt

```
pandas>=2.0
numpy>=1.24
matplotlib>=3.6
seaborn>=0.12
scikit-learn>=1.2
Pillow>=9.0
```

### Reproducējamība

Notebook darbojas lokāli (`../custom/`) un Google Colab (auto-klonē repo).  
Izpildi: **Kernel → Restart & Run All** — visas šūnas jāizpildas bez kļūdām.  
Prezentācijas slaidi tiek ģenerēti automātiski un saglabāti `presentation_slides/` mapē.

In [ ]:
# requirements.txt
reqs = '''pandas>=2.0
numpy>=1.24
matplotlib>=3.6
seaborn>=0.12
scikit-learn>=1.2
Pillow>=9.0
'''

req_path = 'requirements.txt' if IN_COLAB else '../requirements.txt'
with open(req_path, 'w') as f:
    f.write(reqs)
print('requirements.txt atjaunināts (pievienots Pillow>=9.0).')